# Pharmaceutical Information RAG Chatbot

## Project Overview

This project develops a **Retrieval-Augmented Generation (RAG) chatbot** for accessing and answering questions about pharmaceutical products using a collection of structured JSON drug records.

The system combines **semantic search, text embeddings, and a Large Language Model (LLM)** to provide users with relevant pharmaceutical information without requiring the LLM to rely solely on its internal knowledge. Instead, when a user asks a question, the system first retrieves the most relevant pharmaceutical records from the dataset and then provides those records as context to the LLM, which generates the final response.



## Dataset

The knowledge base consists of pharmaceutical drug records stored in JSON files. Each record contains information such as:

* Registration number
* Trade name
* Generic name
* Strength and dosage
* Administration route
* Pharmaceutical form
* Package size
* Legal classification
* Shelf life
* Storage conditions
* Manufacturer
* Marketing company
* Patient Information Leaflet (PIL) in English
* Patient Information Leaflet (PIL) in Arabic
* Summary of Product Characteristics (SPC)

The JSON records are converted into text so that they can be processed by the embedding model and used as retrieval context.

## RAG Pipeline

The system follows five main stages:

**1. Data Loading**

The pharmaceutical JSON files are loaded and converted into individual drug documents.

**2. Text Representation**

Each drug's structured information, PIL content, and SPC information are combined into a textual representation.

**3. Embedding Generation**

The `all-MiniLM-L6-v2` Sentence Transformer model converts each pharmaceutical document into a numerical vector (embedding).

**4. Semantic Retrieval**

When the user submits a question, the question is also converted into an embedding. The system calculates the **cosine similarity** between the question embedding and all pharmaceutical document embeddings. The documents with the highest similarity scores are retrieved as the most relevant context.

**5. Response Generation**

The retrieved pharmaceutical information is passed to an LLM through OpenRouter. The LLM uses the retrieved context to generate an answer while being instructed not to invent information that is not present in the available documents.

## System Architecture

```text
                    Pharmaceutical JSON Files
                              │
                              ▼
                       Data Processing
                              │
                              ▼
                    Text Representation
                              │
                              ▼
                    Document Embeddings
                 Sentence Transformer Model
                    (all-MiniLM-L6-v2)
                              │
                              ▼
User Question ───────► Query Embedding
                              │
                              ▼
                    Cosine Similarity
                              │
                              ▼
                     Top-K Documents
                              │
                              ▼
                    Retrieved Context
                              │
                              ▼
                    OpenRouter LLM
                              │
                              ▼
                     Generated Answer
```




**1. Data Loading**

The pharmaceutical JSON files are loaded and converted into individual drug documents

In [23]:
import json
from pathlib import Path


def load_drugs(folder):

    drugs = []

    json_files = list(folder.glob("*.json"))

    print(f"Found {len(json_files)} JSON files")

    for file in json_files:

        print(f"\nLoading: {file.name}")

        try:

            with open(file, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Each file is a dictionary
            # registration_number -> drug_record

            if not isinstance(data, dict):
                print(" Expected a dictionary")
                continue

            print("  Drugs in file:", len(data))

            # ----------------------------------------
            # Each key is the registration number
            # Each value is the complete drug record
            # ----------------------------------------

            for registration_number, drug_record in data.items():

                if not isinstance(drug_record, dict):
                    print(
                        f" Skipping {registration_number}"
                    )
                    continue

                # Make sure Drug Data exists
                drug_data = drug_record.get(
                    "Drug Data",
                    {}
                )

                # Store everything
                drugs.append({
                    "id": str(registration_number),
                    "data": drug_record
                })

        except Exception as e:

            print(
                f" Error reading {file.name}: {e}"
            )

    return drugs

In [24]:
DATA_FOLDER = Path("/content/drive/MyDrive/Drug data/")

drugs = load_drugs(DATA_FOLDER)

print("\n==============================")
print("TOTAL DRUGS LOADED:", len(drugs))
print("==============================")

Found 563 JSON files

Loading: page_042.cleaned.json
  Drugs in file: 15

Loading: page_020.cleaned.json
  Drugs in file: 15

Loading: page_011.cleaned.json
  Drugs in file: 15

Loading: page_023.cleaned.json
  Drugs in file: 15

Loading: page_052.cleaned.json
  Drugs in file: 15

Loading: page_040.cleaned.json
  Drugs in file: 15

Loading: page_037.cleaned.json
  Drugs in file: 15

Loading: page_024.cleaned.json
  Drugs in file: 15

Loading: page_069.cleaned.json
  Drugs in file: 15

Loading: page_044.cleaned.json
  Drugs in file: 15

Loading: page_014.cleaned.json
  Drugs in file: 15

Loading: page_039.cleaned.json
  Drugs in file: 15

Loading: page_021.cleaned.json
  Drugs in file: 15

Loading: page_060.cleaned.json
  Drugs in file: 15

Loading: page_053.cleaned.json
  Drugs in file: 15

Loading: page_007.cleaned.json
  Drugs in file: 15

Loading: page_003.cleaned.json
  Drugs in file: 15

Loading: page_051.cleaned.json
  Drugs in file: 15

Loading: page_046.cleaned.json
  Drugs in 

**2. Text Representation**

Each drug's structured information, PIL content, and SPC information are combined into a textual representation.

In [25]:
def drug_to_text(drug):

    registration_number = drug["id"]
    data = drug["data"]

    drug_data = data.get(
        "Drug Data",
        {}
    )

    english_pil = data.get(
        "Patient Information Leaflet (PIL) in English",
        ""
    )

    arabic_pil = data.get(
        "Patient Information Leaflet (PIL) in Arabic",
        ""
    )

    spc = data.get(
        "Summary of Product Characteristics (SPC)",
        ""
    )

    text = f"""
Registration Number:
{registration_number}

Drug Data:
{json.dumps(
    drug_data,
    ensure_ascii=False,
    indent=2
)}

Patient Information Leaflet (English):
{english_pil}

Patient Information Leaflet (Arabic):
{arabic_pil}

Summary of Product Characteristics:
{spc}
"""

    return text.strip()

In [26]:
documents = [
    drug_to_text(drug)
    for drug in drugs
]

print("Number of documents:", len(documents))

print("\nFirst document preview:")
print(documents[3][:3000])

Number of documents: 8444

First document preview:
Registration Number:
0511246148

Drug Data:
"{\n  \"Registration Number\": \"0511246148\",\n  \"Register Year\": \"2024\",\n  \"Trade Name\": \"Restomap\",\n  \"Generic Name\": \"ESOMEPRAZOLE MAGNESIUM\",\n  \"Strength\": \"40\",\n  \"Strength Unit\": \"mg\",\n  \"Administration Route\": \"Oral use\",\n  \"Pharmaceutical Form\": \"Gastro-resistant tablet\",\n  \"Package Size\": \"28\",\n  \"Packages Types\": \"Blister\",\n  \"Legal Classification\": \"Prescription\",\n  \"Product Control\": \"Uncontrolled\",\n  \"Drug Type\": \"Generic\",\n  \"ShelfLife in Months\": \"24\",\n  \"Storage Conditions\": \"store below 30°c\",\n  \"Public price (SAR)\": \"52.55\",\n  \"Manufacture\": \"Lee Pharma\",\n  \"الوكيل\": \"Pharma Pharmaceutical Industries (PPI)\",\n  \"Marketing Company\": \"Pharma Pharmaceutical Industries & Biological Products\",\n  \"Anatomical Therapeutic Chemical Code 1\": \"A02BC05\"\n}"

Patient Information Leaflet (English

**3. Embedding Generation**

The all-MiniLM-L6-v2 Sentence Transformer model converts each pharmaceutical document into a numerical vector (embedding).

In [27]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded.


In [28]:
document_embeddings = embedding_model.encode(
    documents,
    normalize_embeddings=True,
    show_progress_bar=True
)

print("Embedding shape:")
print(document_embeddings.shape)

Batches:   0%|          | 0/264 [00:00<?, ?it/s]

Embedding shape:
(8444, 384)


**4. Semantic Retrieval**

When the user submits a question, the question is also converted into an embedding. The system calculates the cosine similarity between the question embedding and all pharmaceutical document embeddings. The documents with the highest similarity scores are retrieved as the most relevant context.

In [29]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity


def retrieve_drugs(query, top_k=3):

    # Embed the user's question
    query_embedding = embedding_model.encode(
        [query],
        normalize_embeddings=True
    )

    # Compare question against every drug
    similarities = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Highest similarity first
    top_indices = np.argsort(
        similarities
    )[::-1][:top_k]

    results = []

    for index in top_indices:

        results.append({
            "drug_id": drugs[index]["id"],
            "score": float(similarities[index]),
            "drug": drugs[index],
            "document": documents[index]
        })

    return results

**5. Response Generation**

The retrieved pharmaceutical information is passed to an LLM through OpenRouter. The LLM uses the retrieved context to generate an answer while being instructed not to invent information that is not present in the available documents

In [30]:
from openai import OpenAI
from google.colab import userdata


api_key = userdata.get("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=api_key
)

MODEL = "inclusionai/ling-3.0-flash-fin:free"

In [31]:
def ask_pharmaceutical_chatbot(
    question,
    top_k=3
):

    # ============================================
    # STEP 1: RETRIEVE
    # ============================================

    results = retrieve_drugs(
        question,
        top_k=top_k
    )

    # ============================================
    # STEP 2: BUILD CONTEXT
    # ============================================

    context_parts = []

    for result in results:

        context_parts.append(
            f"""
RELEVANT DRUG
============

Drug ID:
{result["drug_id"]}

Similarity:
{result["score"]:.3f}

{result["document"]}
"""
        )

    context = "\n\n---------------------------\n\n".join(
        context_parts
    )

    # ============================================
    # STEP 3: PROMPT THE LLM
    # ============================================

    system_prompt = f"""
You are a pharmaceutical information assistant.

Your job is to answer questions using ONLY the
pharmaceutical documents provided below.

Do NOT invent information.

If the answer cannot be found in the provided
documents, say:

"I could not find this information in the
available pharmaceutical documents."

When answering, clearly identify the relevant
drug by its trade name and generic name when
available.

For storage questions, report the exact storage
condition from the source document.

For medical or safety-related questions,
distinguish between information for patients
and information intended for healthcare
professionals.

PHARMACEUTICAL DOCUMENTS
========================

{context}
"""

    # ============================================
    # STEP 4: CALL LLM
    # ============================================

    response = client.chat.completions.create(

        model=MODEL,

        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ],

        extra_body={
            "reasoning": {
                "enabled": False
            }
        }
    )

    # ============================================
    # STEP 5: RETURN ANSWER
    # ============================================

    return response.choices[0].message.content

In [32]:
print("=" * 60)
print("PHARMACEUTICAL RAG CHATBOT")
print("Type /end to stop")
print("=" * 60)

while True:

    question = input("\nYou: ")

    if question.strip().lower() == "/end":
        print("Goodbye!")
        break

    try:

        answer = ask_pharmaceutical_chatbot(
            question,
            top_k=3
        )

        print("\nAI:")
        print(answer)

    except Exception as e:

        print("\nERROR:")
        print(e)

PHARMACEUTICAL RAG CHATBOT
Type /end to stop

You: How do I store the Panadol?

AI:
Based on the available pharmaceutical documents, here is how to store Panadol:

**Panadol 500 mg Film-Coated Tablets (Registration No: 5-288-99):**
- **Storage Condition:** Store below 30°C
- Keep out of the reach and sight of children
- Do not use after the expiry date

**Panadol Soluble 500 mg Tablets (Registration No: 3006222293):**
- **Storage Condition:** Store below 25°C

**Summary from the Product Characteristics:**
- Store below 30°C (for the film-coated tablet formulation)
- Store in a dry place below 30°C
- Keep in the original package

The key storage instruction for Panadol is to **store below 30°C** in a cool, dry place, away from children. If you have the soluble tablet formulation, the temperature requirement is slightly stricter at **below 25°C**.

I could not find this information in the available pharmaceutical documents if you were referring to a different Panadol product.

You: What 